[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jgcalvo/SP1306/blob/main/edps-elipticas/MDF_test.ipynb)

# Diferencias finitas para la ecuación de Poisson en 2D

Traducción de `MDF_v1_test.m`.

Resuelve, con condiciones de frontera homogéneas,

$$-\Delta u = f \quad \text{en } \Omega=(0,1)\times(0,1), \qquad u = 0 \ \text{ sobre } \partial\Omega$$

**Discretización.** Malla uniforme de $M$ intervalos por dirección, $h = 1/M$,
con nodos $(x_i, y_j) = (ih, jh)$. La fórmula de 5 puntos aproxima

$$-\Delta u \;\approx\; \frac{4u_{i,j} - u_{i-1,j} - u_{i+1,j} - u_{i,j-1} - u_{i,j+1}}{h^2}$$

Multiplicando la ecuación por $h^2$ se evita dividir, y el sistema queda

$$4u_{i,j} - u_{i-1,j} - u_{i+1,j} - u_{i,j-1} - u_{i,j+1} = h^2 f_{i,j}$$

Las incógnitas son solo los nodos interiores, $i,j = 1,\dots,M-1$: los de la
frontera valen 0 y por eso no entran al sistema. En total $(M-1)^2$ incógnitas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve

## Ensamblaje de la matriz

En vez de escribir `A[fila, columna] = valor` una entrada a la vez —que en una
matriz rala es caro— se acumulan tres listas paralelas: `ii` (filas), `jj`
(columnas) y `ss` (valores). Cada terna `(ii[k], jj[k], ss[k])` dice *"en la
fila `ii[k]`, columna `jj[k]`, va el valor `ss[k]`"*. Al final,
`coo_matrix((ss, (ii, jj)))` construye la matriz de una sola vez.

Es exactamente lo que hace `sparse(ii,jj,ss,dimA,dimA)` en MATLAB.

**Sobre los índices.** Se conservan los índices matemáticos `i, j = 1..M-1`,
para que la evaluación `f(i*h, j*h)` se lea igual que en el papel. Como Python
numera desde 0, la fila del sistema lleva un `-1` extra:

$$\texttt{pos} = (j-1)(M-1) + (i-1)$$

Los desplazamientos a los vecinos no cambian: $\pm 1$ en $x$, $\pm (M-1)$ en $y$.

## Entradas

In [ ]:
uex = lambda x, y: np.sin(2*np.pi*x)*np.sin(np.pi*y)          # sol. exacta
f   = lambda x, y: 5*np.pi**2*np.sin(2*np.pi*x)*np.sin(np.pi*y)  # lado derecho

M    = 32                # cantidad intervalos
h    = 1/M               # paso de la malla
dimA = (M-1)**2          # cantidad de nodos interiores

## Recorrido de los nodos interiores

`ii`, `jj` y `ss` son **listas de Python**. A diferencia de MATLAB, agregarles
un elemento con `append` no copia la lista: Python reserva espacio de más por
adelantado, así que el costo es constante y no hace falta preasignar.

(La traducción literal de `ii = [ii, pos]` sería `np.append`, y ésa **sí** copia
el arreglo entero en cada paso. En el notebook de convergencia se mide el
desastre que produce.)

In [ ]:
ii, jj, ss = [], [], []                  # formato triplete: filas, columnas, valores
b      = np.zeros(dimA)                  # lado derecho del sistema
UexMat = np.zeros((M-1, M-1))            # sol. exacta evaluada en la malla

for j in range(1, M):                    # j avanza en y
    for i in range(1, M):                # i avanza en x
        pos = (j-1)*(M-1) + (i-1)        # fila del sistema para el nodo (i,j)
        b[pos] = h**2 * f(i*h, j*h)      # lado derecho, ya multiplicado por h^2
        UexMat[i-1, j-1] = uex(i*h, j*h) # referencia para medir el error

        # la fila pos es la ecuacion del nodo (i,j): todas sus entradas se
        # escriben como (fila pos, columna del vecino)

        # termino diagonal: el propio nodo (i,j)
        ii.append(pos); jj.append(pos); ss.append(4.0)

        # vecino izquierdo (i-1,j); si i==1 cae en la frontera x=0, donde u=0,
        # asi que no aporta al sistema
        if i != 1:
            ii.append(pos); jj.append(pos-1); ss.append(-1.0)

        # vecino derecho (i+1,j); si i==M-1 cae en la frontera x=1
        if i != M-1:
            ii.append(pos); jj.append(pos+1); ss.append(-1.0)

        # vecino inferior (i,j-1); si j==1 cae en la frontera y=0
        if j != 1:
            ii.append(pos); jj.append(pos-(M-1)); ss.append(-1.0)

        # vecino superior (i,j+1); si j==M-1 cae en la frontera y=1
        if j != M-1:
            ii.append(pos); jj.append(pos+(M-1)); ss.append(-1.0)

print(f'{dimA} incognitas, {len(ss)} entradas no nulas')

## Armar la matriz y resolver

`coo_matrix` es el formato de tripletes; se convierte a `csr` porque es el que
sabe resolver sistemas. `spsolve` es el equivalente de `A\b` de MATLAB.

In [ ]:
A  = coo_matrix((ss, (ii, jj)), shape=(dimA, dimA)).tocsr()
uh = spsolve(A, b)

## Visualización

`Z` se arma sobre la malla completa, incluyendo la frontera: los ceros del borde
**son** la condición de frontera.

El `reshape` con `order='F'` apila por columnas, igual que MATLAB, y la
transpuesta es necesaria porque `meshgrid` indexa `Z[fila, columna] = Z[y, x]`,
al revés que `UexMat[i,j] = u(x_i, y_j)`.

In [ ]:
xx = np.linspace(0, 1, M+1)              # malla completa, con frontera
X, Y = np.meshgrid(xx, xx)
Z = np.zeros((M+1, M+1))
Z[1:-1, 1:-1] = uh.reshape((M-1, M-1), order='F').T

fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(projection='3d')
ax.plot_surface(X, Y, Z, cmap='viridis', linewidth=0, antialiased=False)
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.show()

## Error

`UexMat.ravel(order='F')` apila por columnas, que es justo el orden
`pos = (j-1)(M-1)+(i-1)` con que se armó `uh`: los dos vectores son comparables
entrada a entrada.

In [ ]:
error_inf = np.linalg.norm(uh - UexMat.ravel(order='F'), np.inf)
error_2   = np.linalg.norm(uh - UexMat.ravel(order='F'))
print(f'error en norma infinito: {error_inf:.6e}')
print(f'error en norma 2       : {error_2:.6e}')